# setup

In [2]:
%%capture
"""
try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
except: _numpy = "numpy"; _pil = "pillow"
!uv pip install -qqq \
    "torch>=2.8.0" "triton>=3.4.0" {_numpy} {_pil} torchvision bitsandbytes \
    unsloth "unsloth_zoo>=2026.4.6" transformers==5.5.0 torchcodec timm
"""

In [3]:
#!uv pip install datasets

Add Lora adapters

# Create SFT dataset

In [4]:
# mix of dataset

In [5]:
from datasets import load_dataset

/Users/a415137/personal_projects/smol_course/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
alpaca_dataset = load_dataset('vicgalle/alpaca-gpt4', split='train[:10000]')

In [7]:
alpaca_dataset

Dataset({
    features: ['instruction', 'input', 'output', 'text'],
    num_rows: 10000
})

In [8]:
finetome_dataset = load_dataset("mlabonne/FineTome-100k", split = "train[:3000]")

In [9]:
finetome_dataset

Dataset({
    features: ['conversations', 'source', 'score'],
    num_rows: 3000
})

In [10]:
everyday_convos = load_dataset("HuggingFaceTB/everyday-conversations-llama3.1-2k", split='train_sft')

In [11]:
everyday_convos

Dataset({
    features: ['topic', 'subtopic', 'subsubtopic', 'full_topic', 'prompt', 'completion', 'token_length', 'messages'],
    num_rows: 2260
})

In [12]:
everyday_convos[0]['messages']

[{'content': 'Hey!', 'role': 'user'},
 {'content': 'Hello! How can I help you today?', 'role': 'assistant'},
 {'content': "I'm trying to track my expenses. Can you help me with that?",
  'role': 'user'},
 {'content': 'Yes, I can help you track your expenses. You can start by telling me your income and fixed expenses, such as rent and utilities.',
  'role': 'assistant'},
 {'content': "That sounds easy. How do I know if I'm staying within my budget?",
  'role': 'user'},
 {'content': "Once you've entered your income and expenses, I can help you set a budget and alert you when you're going over. You can also categorize your spending to see where your money is going.",
  'role': 'assistant'},
 {'content': 'Okay, that sounds great. Can you remind me to review my budget regularly?',
  'role': 'user'},
 {'content': 'I can send you reminders to review your budget weekly, monthly, or at any interval you prefer.',
  'role': 'assistant'}]

# formatting datasets for SFT

In [13]:
alpaca_dataset[0]

{'instruction': 'Give three tips for staying healthy.',
 'input': '',
 'output': '1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.\n\n2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.\n\n3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.',
 'text': 'Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n### Instruction:\nGive three tips for 

In [14]:
alpaca_dataset[0]['instruction']

'Give three tips for staying healthy.'

In [15]:
print(alpaca_dataset[0]['output'])

1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.

2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.

3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.


In [16]:
# format alpaca to match the chat template format
def create_conversation_alpaca(example):
    return {
        "messages": [
            {
                "role": "user",
                "content": example['instruction']},
            {
                "role": "assistant",
                "content": example['output']
            },
        ]
    }

In [17]:
alpaca_dataset

Dataset({
    features: ['instruction', 'input', 'output', 'text'],
    num_rows: 10000
})

In [18]:
#alpaca_dataset[100]

In [19]:
alpaca_dataset_formatted = alpaca_dataset.map(create_conversation_alpaca, batched=False)

In [20]:
alpaca_dataset_formatted['messages'][0]

[{'content': 'Give three tips for staying healthy.', 'role': 'user'},
 {'content': '1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.\n\n2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.\n\n3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.',
  'role': 'assistant'}]

# format finetome

In [21]:
finetome_dataset['conversations'][1]

[{'from': 'human',
  'value': 'Explain how recursion works and provide a recursive function in Python that calculates the factorial of a given number.'},
 {'from': 'gpt',
  'value': "Recursion is a programming technique where a function calls itself to solve a problem. It breaks down a complex problem into smaller, more manageable subproblems until a base case is reached. The base case is a condition where the function does not call itself, but instead returns a specific value or performs a specific action.\n\nIn the case of calculating the factorial of a number, recursion can be used to break down the problem into simpler subproblems. The factorial of a non-negative integer n is the product of all positive integers less than or equal to n.\n\nHere is a recursive function in Python that calculates the factorial of a given number:\n\n```python\ndef factorial(n):\n    # Base case: factorial of 0 or 1 is 1\n    if n == 0 or n == 1:\n        return 1\n    # Recursive case: factorial of n i

In [22]:
users = [ex['from'] for row in finetome_dataset['conversations'] for ex in row]

In [23]:
users = set(users)
users

{'gpt', 'human', 'system'}

In [24]:
finetome_dataset['conversations'][15]

[{'from': 'system',
  'value': 'You are a helpful assistant, with no access to external functions.'},
 {'from': 'human', 'value': 'Explain the comparison operator in programming.'},
 {'from': 'gpt',
  'value': 'In programming, a comparison operator is used to compare two values and determine if they are equal or if one is greater or less than the other. It is usually denoted by symbols such as "==", ">", "<", ">=", "<=", or "!=". \n\nThe "==" symbol is used to check if two values are equal, while "!=" checks if they are not equal. The ">" symbol checks if the left value is greater than the right value, while "<" checks if the left value is less than the right value. The ">=" symbol checks if the left value is greater than or equal to the right value, while "<=" checks if the left value is less than or equal to the right value.\n\nComparison operators are often used in decision-making structures such as if statements and loops to determine which code block should be executed based on th

In [25]:
def create_conversation_finetome(example):
    dict_users = {
        "human": "user",
        "gpt": "assistant"
    }
    conversation = example['conversations']
    return {
        "messages": [
            {
                "role": dict_users[ex['from']],
                "content": ex['value']
            } for ex in conversation
            if ex['from'] in dict_users
        ]
    }

In [26]:
finetome_dataset_formatted = finetome_dataset.map(create_conversation_finetome, batched=False)

In [27]:
finetome_dataset_formatted['messages']

Column([[{'content': 'Explain what boolean operators are, what they do, and provide examples of how they can be used in programming. Additionally, describe the concept of operator precedence and provide examples of how it affects the evaluation of boolean expressions. Discuss the difference between short-circuit evaluation and normal evaluation in boolean expressions and demonstrate their usage in code. \n\nFurthermore, add the requirement that the code must be written in a language that does not support short-circuit evaluation natively, forcing the test taker to implement their own logic for short-circuit evaluation.\n\nFinally, delve into the concept of truthiness and falsiness in programming languages, explaining how it affects the evaluation of boolean expressions. Add the constraint that the test taker must write code that handles cases where truthiness and falsiness are implemented differently across different programming languages.', 'role': 'user'}, {'content': 'Boolean operat

# everyday conversations

In [28]:
list(everyday_convos['messages'])

[[{'content': 'Hey!', 'role': 'user'},
  {'content': 'Hello! How can I help you today?', 'role': 'assistant'},
  {'content': "I'm trying to track my expenses. Can you help me with that?",
   'role': 'user'},
  {'content': 'Yes, I can help you track your expenses. You can start by telling me your income and fixed expenses, such as rent and utilities.',
   'role': 'assistant'},
  {'content': "That sounds easy. How do I know if I'm staying within my budget?",
   'role': 'user'},
  {'content': "Once you've entered your income and expenses, I can help you set a budget and alert you when you're going over. You can also categorize your spending to see where your money is going.",
   'role': 'assistant'},
  {'content': 'Okay, that sounds great. Can you remind me to review my budget regularly?',
   'role': 'user'},
  {'content': 'I can send you reminders to review your budget weekly, monthly, or at any interval you prefer.',
   'role': 'assistant'}],
 [{'content': 'Hi', 'role': 'user'},
  {'con

In [29]:
sft_dataset = [list(alpaca_dataset_formatted['messages']), list(finetome_dataset_formatted['messages']), list(everyday_convos['messages'])]

In [30]:
sft_dataset = [s for sublist in sft_dataset for s in sublist]
sft_dataset

[[{'content': 'Give three tips for staying healthy.', 'role': 'user'},
  {'content': '1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.\n\n2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.\n\n3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.',
   'role': 'assistant'}],
 [{'content': 'What are the three primary colors?', 'role': 'user'},
  {'content': 'The three primary colors are red, blue, and

In [31]:
# convert to dataset
from datasets import Dataset
sft_data = Dataset.from_dict({"messages": sft_dataset})

In [32]:
sft_data

Dataset({
    features: ['messages'],
    num_rows: 15260
})

In [33]:
sft_data.save_to_disk('sft_data')

Saving the dataset (1/1 shards): 100%|██████████| 15260/15260 [00:00<00:00, 1305373.61 examples/s]


NameError: name 'a' is not defined

# Apply chat template

## Alpaca dataset

In [ ]:
alpaca_dataset[10]

{'instruction': 'How did Julius Caesar die?',
 'input': '',
 'output': "Julius Caesar, the Roman Military general, and statesman was assassinated on March 15, 44 BC. He was stabbed to death at a meeting of the Senate in the Theatre of Pompey by a group of senators who called themselves the Liberatores. Led by Gaius Cassius Longinus, and Marcus Junius Brutus, they believed that Caesar's dictatorship was subverting the Roman Republic and sought to restore the traditional republican government. The exact number of assailants is not known, but it is believed that a group of as many as 60 conspirators participated in the assassination, and each of them took turns in stabbing Caesar. The day of his death is still known as the Ides of March.",
 'text': "Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n### Instruction:\nHow did Julius Caesar die?\n\n### Response:\nJulius Caesar, the Roman Military general, and statesman was assassinat

In [ ]:
def format_alpaca_dataset(example):
    convos = example['messages']
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False).removeprefix('<bos>') for convo in convos]
    return { "text" : texts, }

In [ ]:
alpaca_txt = alpaca_dataset.map(format_alpaca_dataset, batched=True)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [ ]:
print(alpaca_txt['text'][10])

<|turn>user
How did Julius Caesar die?<turn|>
<|turn>model
Julius Caesar, the Roman Military general, and statesman was assassinated on March 15, 44 BC. He was stabbed to death at a meeting of the Senate in the Theatre of Pompey by a group of senators who called themselves the Liberatores. Led by Gaius Cassius Longinus, and Marcus Junius Brutus, they believed that Caesar's dictatorship was subverting the Roman Republic and sought to restore the traditional republican government. The exact number of assailants is not known, but it is believed that a group of as many as 60 conspirators participated in the assassination, and each of them took turns in stabbing Caesar. The day of his death is still known as the Ides of March.<turn|>



## Finetome txt dataset

In [ ]:
finetome_dataset[0]

{'conversations': [{'role': 'user',
   'content': 'Explain what boolean operators are, what they do, and provide examples of how they can be used in programming. Additionally, describe the concept of operator precedence and provide examples of how it affects the evaluation of boolean expressions. Discuss the difference between short-circuit evaluation and normal evaluation in boolean expressions and demonstrate their usage in code. \n\nFurthermore, add the requirement that the code must be written in a language that does not support short-circuit evaluation natively, forcing the test taker to implement their own logic for short-circuit evaluation.\n\nFinally, delve into the concept of truthiness and falsiness in programming languages, explaining how it affects the evaluation of boolean expressions. Add the constraint that the test taker must write code that handles cases where truthiness and falsiness are implemented differently across different programming languages.'},
  {'role': 'as

In [ ]:
def format_finetome_dataset(example):
    convos = example['conversations']
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False).removeprefix('<bos>') for convo in convos]
    return { "text" : texts, }

In [ ]:
txt_finetome = finetome_dataset.map(format_finetome_dataset, batched=True)

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

In [ ]:
print(txt_finetome['text'][20])

<|turn>user
Write a program to find the number of letters in each word of a sentence using the map function.<turn|>
<|turn>model
Certainly! Here's a code example that uses the `map()` function to find the number of letters in each word of a sentence:

```python
def count_letters(sentence):
    """
    This function takes a sentence as input and returns a list of the number of letters
    in each word using the map function.

    Args:
        sentence (str): The input sentence containing words.

    Returns:
        list: A list of integers representing the number of letters in each word.

    Examples:
        >>> count_letters("Hello world")
        [5, 5]
        >>> count_letters("Python is awesome")
        [6, 2, 7]
    """
    # Split the sentence into words
    words = sentence.split()

    # Use map to apply len() function on each word and get the length
    letter_counts = list(map(len, words))

    return letter_counts
```

Now, let me explain the code:

1. The `count_letter

## everyday convos

In [ ]:
everyday_convos[0]

{'topic': 'Shopping',
 'subtopic': 'Budgeting',
 'subsubtopic': 'Tracking expenses',
 'full_topic': 'Shopping/Budgeting/Tracking expenses',
 'prompt': 'Generate a very simple multi-turn conversation between a User and an AI Assistant about Shopping/Budgeting/Tracking expenses. The conversation should start with a basic greeting like "Hello" or "Hi" and be straightforward. Include 3-4 short exchanges. The AI should give brief, clear answers. The User should ask simple questions.\n\nStart the conversation like this:\n\nUser: [Greeting]\n\nAI: Hello! How can I help you today?\n\nUser: [Continue with a simple question or statement]\n\nAI: [Respond briefly and clearly]\n\nUser: [Ask a follow-up question or make another simple statement]\n\nAI: [Provide a final helpful response]\n\nMake sure the entire conversation remains very simple and easy to understand, focusing on basic topics or requests.',
 'completion': "User: Hi\n\nAI: Hello! How can I help you today?\n\nUser: I'm trying to track m

In [ ]:
def format_everyday_convos(example):
    convos = example['messages']
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False).removeprefix('<bos>') for convo in convos]
    return { "text" : texts, }

In [ ]:
txt_everyday = everyday_convos.map(format_everyday_convos, batched=True)

Map:   0%|          | 0/2260 [00:00<?, ? examples/s]

In [ ]:
print(txt_everyday['text'][0])

<|turn>user
Hey!<turn|>
<|turn>model
Hello! How can I help you today?<turn|>
<|turn>user
I'm trying to track my expenses. Can you help me with that?<turn|>
<|turn>model
Yes, I can help you track your expenses. You can start by telling me your income and fixed expenses, such as rent and utilities.<turn|>
<|turn>user
That sounds easy. How do I know if I'm staying within my budget?<turn|>
<|turn>model
Once you've entered your income and expenses, I can help you set a budget and alert you when you're going over. You can also categorize your spending to see where your money is going.<turn|>
<|turn>user
Okay, that sounds great. Can you remind me to review my budget regularly?<turn|>
<|turn>model
I can send you reminders to review your budget weekly, monthly, or at any interval you prefer.<turn|>



Concatenate all the txt dataset

In [ ]:
txt_list = [alpaca_txt['text'], txt_finetome['text'], txt_everyday['text']]

In [ ]:
#txt_list 

In [ ]:
from datasets import Dataset

In [ ]:
from datasets import Dataset, concatenate_datasets

combined_dataset = concatenate_datasets([
    Dataset.from_dict({"text": alpaca_txt['text']}),
    Dataset.from_dict({"text": txt_finetome['text']}),
    Dataset.from_dict({"text": txt_everyday['text']}),
])
combined_dataset

Dataset({
    features: ['text'],
    num_rows: 15260
})

In [ ]:
combined_dataset.save_to_disk("../gemma_data/gemma_sft_dataset")

Saving the dataset (0/1 shards):   0%|          | 0/15260 [00:00<?, ? examples/s]

In [ ]:
combined_dataset = combined_dataset.shuffle(seed=42)

In [ ]:
#print(combined_dataset[100]['text'])

In [ ]:
split_data =combined_dataset.train_test_split(test_size=0.1)

In [ ]:
train_data = split_data['train']
eval_data = split_data['test']

In [ ]:
train_data

Dataset({
    features: ['text'],
    num_rows: 13734
})

In [ ]:
train_data = train_data.select(range(3000))

In [ ]:
ls

sample_data/  unsloth_compiled_cache/


# Train the model

In [ ]:
model.config.bos_token_id = tokenizer.bos_token_id
model.generation_config.bos_token_id = tokenizer.bos_token_id

In [ ]:
!uv pip install wandb weave

Using Python 3.12.13 environment at: /usr
Checked 2 packages in 95ms


In [ ]:
import os
os.environ['WANDB_API_KEY'] = 'your_api_key'


In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_data,
    eval_dataset = eval_data, # Can set up evaluation!
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 1,
        per_device_eval_batch_size = 1,
        gradient_accumulation_steps = 4, # Use GA to mimic batch size!
        #eval_accumulation_steps = 4,
        #save_strategy = "steps",
        #save_total_limit = 3,
        #save_steps=60,
        eval_strategy="steps",
        eval_steps=20,
        warmup_steps = 5,
        num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 180,
        learning_rate = 2e-5, # Reduce to 2e-5 for long training runs
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "wandb", # Use TrackIO/WandB etc
    ),
)

Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/3000 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/1526 [00:00<?, ? examples/s]

In [ ]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|turn>user\n",
    response_part = "<|turn>model\n",
)

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1526 [00:00<?, ? examples/s]

In [ ]:
tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[100]["labels"]]).replace(tokenizer.pad_token, " ")

'            Blockchain is a decentralized, distributed digital ledger technology that records transactions in a way that is transparent, secure, and tamper-proof. At its core, a blockchain is a chain of blocks that contains information about transactions. Each block in the chain is made up of a list of transactions, a timestamp, and a cryptographic hash of the previous block. The blocks are linked together in a chain, with each new block being added to the end of the chain.\n\nThe decentralized nature of the blockchain means that it does not rely on any single entity or authority to validate and verify transactions. Instead, transactions are validated and verified by a network of users or nodes that work together to maintain the integrity of the ledger.\n\nBlockchain technology is best known as the underlying technology behind cryptocurrencies like Bitcoin, but it has many other potential applications, from supply chain management to voting systems. Its key features are decentralizati

In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3,000 | Num Epochs = 1 | Total steps = 180
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 12,668,928 of 5,135,846,944 (0.25% trained)


RuntimeError: expected mat1 and mat2 to have the same dtype, but got: float != c10::Half

In [ ]:
!nvidia-smi

In [ ]:
torch.cuda.current_device()

In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

# Inference

In [ ]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-4",
)
messages = [{
    "role": "user",
    "content": [{
        "type" : "text",
        "text" : "Continue the sequence: 1, 1, 2, 3, 5, 8,",
    }]
}]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
    tokenize = True,
    return_dict = True,
).to("cuda")
outputs = model.generate(
    **inputs,
    max_new_tokens = 64, # Increase for longer outputs!
    use_cache = False,
    # Recommended Gemma-4 settings!
    temperature = 1.0, top_p = 0.95, top_k = 64,
)
tokenizer.batch_decode(outputs)

In [ ]:
messages = [{
    "role": "user",
    "content": [{"type" : "text", "text" : "Why is the sky blue?",}]
}]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
    tokenize = True,
    return_dict = True,
).to("cuda")

from transformers import TextStreamer
_ = model.generate(
    **inputs,
    max_new_tokens = 64, # Increase for longer outputs!
    use_cache = False,
    # Recommended Gemma-4 settings!
    temperature = 1.0, top_p = 0.95, top_k = 64,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

To save the final model as LoRA adapters, either use Huggingface's push_to_hub for an online save or save_pretrained for a local save.

[NOTE] This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [ ]:
model_save_path="gemma-4-finetune"

In [ ]:
if True: # Change to True to save finetune!
    model.save_pretrained_merged(model_save_path, tokenizer)

In [ ]:
if True:
    from unsloth import FastModel
    model, tokenizer = FastModel.from_pretrained(
        model_name = model_save_path, # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = 2048,
        load_in_4bit = True,
    )

messages = [{
    "role": "user",
    "content": [{"type" : "text", "text" : "What is Gemma-4?",}]
}]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
    tokenize = True,
    return_dict = True,
).to("cuda")

from transformers import TextStreamer
_ = model.generate(
    **inputs,
    max_new_tokens = 128, # Increase for longer outputs!
    use_cache = False,
    # Recommended Gemma-4 settings!
    temperature = 1.0, top_p = 0.95, top_k = 64,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

In [ ]:
messages = [{
    "role": "user",
    "content": [{ "type" : "text",
                  "text" : "How many r are in strawberry?" }]
}]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
    tokenize = True,
    return_dict = True,
).to("cuda")


from transformers import TextStreamer
_ = model.generate(
    **inputs,
    max_new_tokens = 128, # Increase for longer outputs!
    use_cache = False,
    # Recommended Gemma-4 settings!
    temperature = 1.0, top_p = 0.95, top_k = 64,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)